# AIHub 증강 학습 (run_experiment_augmented)

검증된 baseline(Kaggle train, LB 0.916) 위에 **AIHub 조합 1,3,4,5,6을 추가 학습 데이터로** 사용.

원칙:
1. **val은 Kaggle 대표 검증셋으로 유지** — AIHub는 train에만 추가.
2. **누수 0** — AIHub 조합이 val 조합과 겹치면 자동 제외.
3. **혼합 이미지 박스 보존** — 56외 약 박스도 그대로 학습(지우면 라벨 없는 알약이 생겨 악영향). 56이 하나도 없는 이미지만 제외.
4. **category_id는 K-코드로 통일** — AIHub/Kaggle 동일 체계.
5. **제출은 56 클래스로 제한** — 그 외 예측은 자동 제거.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
from src.joelchoi.data.kaggle_converter import convert_kaggle_to_coco
from src.joelchoi.data.aihub_converter import convert_aihub_to_coco
from src.joelchoi.data.subset import create_subset, combo_group_key
from src.joelchoi.data.merge import merge_for_augmentation
from src.joelchoi.data.coco_to_yolo import prepare_yolo_dataset
from src.joelchoi.train_yolo import run_yolo_experiment
from src.joelchoi.utils import load_config

CONFIGS_DIR = PROJECT_ROOT / "configs" / "joelchoi"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments" / "joelchoi"
DATA_DIR = PROJECT_ROOT / "data" / "joelchoi"
YOLO_DIR = DATA_DIR / "yolo_aug"
print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/joelchoi/study/projects/project1_3team


## 1단계 — Kaggle train → COCO, 대표 검증셋(val) 분할
val은 baseline과 동일하게 유지(점수 비교 가능).

In [3]:
kaggle_coco = convert_kaggle_to_coco()
allowed_ids = {c["id"] for c in kaggle_coco["categories"]}  # 제출 대상 56
print("Kaggle 클래스:", len(allowed_ids))

train_k, val_k = create_subset(kaggle_coco, tier="full", test_size=0.2, seed=42)

Kaggle 학습 데이터 변환 완료: 이미지 232장, 어노테이션 763개, 클래스 56개
Kaggle 클래스: 56
[full] train: 187장(91조합), val: 45장(23조합) | 조합 누수 0


## 2단계 — AIHub 조합 변환 → train에만 증강 병합
`combo_nums`에서 2번(테스트)은 자동 차단됨.

In [ ]:
aihub_coco = convert_aihub_to_coco(combo_nums=[1, 3])  # , 4, 5, 6])

train_aug, val_aug = merge_for_augmentation(
    base_train=train_k,
    base_val=val_k,
    extra_coco=aihub_coco,
    allowed_ids=allowed_ids,
    drop_pure_irrelevant=True,
)

변환 중: TS_01/TL_01 ...
  → 이미지 1188장, 어노테이션 4738개
변환 중: TS_03/TL_03 ...
  → 이미지 1491장, 어노테이션 5676개

총 이미지: 2679, 어노테이션: 10414, 클래스: 80
증강 병합 완료: base train 187 + AIHub 2678 = 2865장 | val 45장(유지)
  제외: 누수 0, 중복 0, 무관(56없음) 1 | 클래스 91개


## 3단계 — YOLO 데이터셋 생성 + 학습

In [5]:
yaml_path = prepare_yolo_dataset(train_aug, val_aug, YOLO_DIR, symlink=True)
CLASS_MAP = YOLO_DIR / "class_map.json"  # 전체 클래스(K-코드) 매핑

cfg = load_config(CONFIGS_DIR / "experiment" / "exp011_yolo11n_aug.yaml")
metrics = run_yolo_experiment(cfg, yaml_path, EXPERIMENTS_DIR)
print(metrics)  # val은 Kaggle 대표셋 → baseline(0.84)과 직접 비교 가능

YOLO yaml 저장: /Users/joelchoi/study/projects/project1_3team/data/joelchoi/yolo_aug/data.yaml
클래스 매핑 저장: /Users/joelchoi/study/projects/project1_3team/data/joelchoi/yolo_aug/class_map.json
YOLO 데이터셋 준비 완료: train 2865장, val 45장
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/joelchoi/study/projects/project1_3team/data/joelchoi/yolo_aug/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300

## 4단계 — 제출용 class_map(56 제한) + 추론 + 제출

In [6]:
from src.joelchoi.inference import run_inference, save_submission, restrict_class_map

EXP_NAME = cfg["experiment"]["name"]
KAGGLE_DATA = (
    Path.home()
    / ".cache/kagglehub/competitions/ai12-level1-project/sprint_ai_project1_data"
)
WEIGHTS = EXPERIMENTS_DIR / EXP_NAME / "weights" / "best.pt"
SUBMISSION = PROJECT_ROOT / "submissions" / "joelchoi" / f"{EXP_NAME}_submission.csv"

# 56 클래스만 남긴 제출용 class_map
CLASS_MAP_SUBMIT = restrict_class_map(CLASS_MAP, allowed_ids)

preds = run_inference(
    weights_path=WEIGHTS,
    class_map_path=CLASS_MAP_SUBMIT,
    test_dir=KAGGLE_DATA / "test_images",
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=cfg["data"]["img_size"],
)
save_submission(preds, SUBMISSION)
print("제출 파일:", SUBMISSION)

제출용 class_map: 56/91개 클래스 유지(나머지 -1) → /Users/joelchoi/study/projects/project1_3team/data/joelchoi/yolo_aug/class_map_submit.json
추론 대상: 842장
총 예측 수: 2812개 (이미지당 평균 3.3개)
제출 파일 저장: /Users/joelchoi/study/projects/project1_3team/submissions/joelchoi/exp011_yolo11n_aug_submission.csv (2812개 예측)
제출 파일: /Users/joelchoi/study/projects/project1_3team/submissions/joelchoi/exp011_yolo11n_aug_submission.csv


## 5단계 — 제출 파일 점검

In [ ]:
import pandas as pd

df = pd.read_csv(SUBMISSION)
test_ids = {int(p.stem) for p in (KAGGLE_DATA / "test_images").glob("*.png")}
pred_ids = set(df["image_id"].unique())
bad = set(df["category_id"].unique()) - allowed_ids
print(
    f"행 {len(df)}, 예측 이미지 {len(pred_ids)}/{len(test_ids)}, 고유 category {df['category_id'].nunique()}"
)
print("56 외 category_id?", bad if bad else "없음(정상)")
df.head()

행 2812, 예측 이미지 842/842, 고유 category 56
56 외 category_id 섭임? 없음(정상)


,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,1900,157,251,202,124,0.99
1,2,1,16551,558,71,396,403,0.96
2,3,1,27926,598,673,255,479,0.96
3,4,1,24850,173,741,181,291,0.88
4,5,3,1900,140,242,199,128,0.99
